In [21]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR) #can imoprt litellm without any warnings

from litellm import completion
import litellm
litellm.suppress_debug_info=True

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Gemini key loaded: ", "✅" if os.getenv("GEMINI_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:     ❌
Gemini key loaded:  ✅
Groq key loaded:       ✅


# Unified API integration

In [11]:
prompt="Explain RAG in a sentence"

providers=[
    ("OpenAI",     "gpt-4.1-nano"),
    ("Groq",       "groq/llama-3.1-8b-instant"),
    ("Anthropic",  "claude-haiku-4-5-20251001"),
    ("Gemini",     "gemini/gemini-3.1-flash-lite"),
]

#one loop+function call and multiple providers
for label,model in providers:
    try:
        r=completion(model=model,messages=[{"role":"user","content":prompt}])
        print(f"{label}:{r.choices[0].message.content[:80]}") #choices is the first replied response from the model while using litellm u can use this!
    except Exception as e:
        print(f"{label}:{type(e).__name__}")

OpenAI:InternalServerError
Groq:RAG is an acronym that stands for Red, Amber, and Green, referring to a traffic 
Anthropic:AuthenticationError
Gemini:Retrieval-Augmented Generation (RAG) is a technique that enhances large language


# Automatic fallback

In [22]:
response = completion(
    model="gpt-4.1-nano",
    messages=[{"role": "user", "content": "What is an LLM"}],
    fallbacks=[
        "gemini/gemini-3.1-flash-lite",
        "groq/llama-3.1-8b-instant"
    ]
)

print(response.choices[0].message.content[:200])
print(response.model)

16:33:13 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gpt-4.1-nano: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.
Traceback (most recent call last):
  File "e:\Agents\agent\Lib\site-packages\litellm\llms\openai\openai.py", line 852, in acompletion
    openai_aclient: AsyncOpenAI = self._get_openai_client(
                                  ^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Agents\agent\Lib\site-packages\litellm\llms\openai\openai.py", line 369, in _get_openai_client
    _new_client: OpenAI | AsyncOpenAI = AsyncOpenAI(
                                        ^^^^^^^^^^^^
  File "e:\Agents\agent\Lib\site-packages\openai\_client.py", line 837, in __init__
    raise OpenAIError(
openai.OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or

Task was destroyed but it is pending!
task: <Task pending name='Task-648' coro=<LoggingWorker._worker_loop() running at e:\Agents\agent\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:111>>


An **LLM** stands for **Large Language Model**. At its simplest, it is a type of artificial intelligence trained to understand, generate, and manipulate human language.

If you have used ChatGPT, Clau
gemini-3.1-flash-lite


# Cost Tracking

In [26]:
from litellm import completion_cost
response = completion(
    model="gpt-4.1-nano",
    messages=[{"role": "user", "content": "What is an LLM"}],
    fallbacks=[
        "gemini/gemini-3.1-flash-lite",
        "groq/llama-3.1-8b-instant"
    ]
)
cost=completion_cost(completion_response=response)

print("Response:", response.choices[0].message.content)
print("\nInput tokens:", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:${cost:.8f}")

16:54:16 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gpt-4.1-nano: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.
Traceback (most recent call last):
  File "e:\Agents\agent\Lib\site-packages\litellm\llms\openai\openai.py", line 852, in acompletion
    openai_aclient: AsyncOpenAI = self._get_openai_client(
                                  ^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Agents\agent\Lib\site-packages\litellm\llms\openai\openai.py", line 369, in _get_openai_client
    _new_client: OpenAI | AsyncOpenAI = AsyncOpenAI(
                                        ^^^^^^^^^^^^
  File "e:\Agents\agent\Lib\site-packages\openai\_client.py", line 837, in __init__
    raise OpenAIError(
openai.OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or

Response: An **LLM** stands for **Large Language Model**.

At its simplest, an LLM is a type of artificial intelligence trained to understand, generate, and manipulate human language. If you have used tools like ChatGPT, Claude, or Gemini, you have interacted with an LLM.

Here is a breakdown of what that name actually means:

### 1. The Breakdown of the Name
*   **Large:** This refers to two things: the massive amount of data the model was trained on (trillions of words from the internet, books, code, and articles) and the number of "parameters" the model has. Parameters are essentially the internal settings or "connections" the AI adjusts during training to learn patterns. Modern LLMs have billions or even trillions of these parameters.
*   **Language:** The primary focus of these models is human communication. They are designed to process grammar, context, nuance, and even specialized languages like computer programming code.
*   **Model:** This is a statistical representation of la

# Caching

In [27]:
import litellm

#Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

#Also clear any router-strategy state
litellm.cache = None

In [32]:
from litellm import Cache
import time
from litellm import completion

litellm.cache=Cache(type="local") #in-memory cache u could use redis in prod!

prompt = "What does LLM stand for? Answer in one line."

# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"First call (API):{t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"Second call (cache):{t2:.4f}s — {r2.choices[0].message.content}")

print(f"\nSpeedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

First call (API):1.42s — LLM stands for Large Language Model.
Second call (cache):0.0048s — LLM stands for Large Language Model.

Speedup: 294.6x faster, and ZERO cost on the second call!


# Smart Routing

In [34]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "gemini/gemini-3.1-flash-lite",
            "api_key": os.getenv("GEMINI_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",
        "litellm_params": {
            "model": "groq/llama-3.1-8b-instant",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
]

router = Router(model_list=model_list)

fast_response=router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response=router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("Fast/cheap (Gemini): ", fast_response.choices[0].message.content[:150])
print("\nSmart/coding (Groq):\n", code_response.choices[0].message.content[:300])

Fast/cheap (Gemini):  AI is fundamentally reshaping software development by automating coding tasks, accelerating deployment cycles, and enabling the creation of more intui

Smart/coding (Groq):
 **Reversing a String in Python**

Here's a simple function to reverse a string in Python:

```python
def reverse_string(s):
    """
    Reverses a given string.

    Args:
        s (str): The string to be reversed.

    Returns:
        str: The reversed string


# Load Balancer-Multiple API KEYS

### 🎯 Strategy: simple-shuffle — the shortest-line pattern

In [36]:
from litellm import Router

model_list=[
    {
        "model_name": "pool",
        "litellm_params": {
            "model": "gemini/gemini-3.1-flash-lite",
            "api_key": os.getenv("GEMINI_API_KEY")
        },
        "model_info": {"id": "gemini"}
    },
    {
        "model_name": "pool",
        "litellm_params": {
            "model": "groq/llama-3.1-8b-instant",
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info": {"id": "groq"}
    },
]
router=Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------


Task was destroyed but it is pending!
task: <Task pending name='Task-1042' coro=<AsyncClient.aclose() running at e:\Agents\agent\Lib\site-packages\httpx\_client.py:1985>>
Task was destroyed but it is pending!
task: <Task pending name='Task-1043' coro=<AsyncClient.aclose() running at e:\Agents\agent\Lib\site-packages\httpx\_client.py:1985>>
Task was destroyed but it is pending!
task: <Task pending name='Task-1044' coro=<AsyncClient.aclose() running at e:\Agents\agent\Lib\site-packages\httpx\_client.py:1985>>
Task was destroyed but it is pending!
task: <Task pending name='Task-1045' coro=<AsyncClient.aclose() running at e:\Agents\agent\Lib\site-packages\httpx\_client.py:1985>>
Task was destroyed but it is pending!
task: <Task pending name='Task-1046' coro=<AsyncClient.aclose() running at e:\Agents\agent\Lib\site-packages\httpx\_client.py:1985>>
Task was destroyed but it is pending!
task: <Task pending name='Task-1047' coro=<AsyncClient.aclose() running at e:\Agents\agent\Lib\site-package

#1        gemini                  5509 ms   Hello! I am ready to assist you. Pl
#2        gemini                  1024 ms   Hello! I am ready to assist you. Pl
#3        groq                     413 ms   Hello. Would you like three suggest
#4        groq                     247 ms   Hello, 4 times.

Hello, 
Hello, 
He
#5        groq                     590 ms   Hello.

You requested 5, is there a
#6        groq                     557 ms   Hello. 

I'd be happy to respond wi


### 🎯 Strategy 1: least-busy — the shortest-line pattern

The router tracks which deployment is currently least busy and sends the next request there.

In [39]:
import os
from litellm import Router
from collections import Counter

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-3.1-flash-lite",
                        "api_key": os.getenv("GEMINI_API_KEY")},
     "model_info": {"id": "🟣 Gemini Flash Lite"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.1-8b-instant",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama 3.1 8B"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

Request 1 → 🟣 Gemini Flash Lite
Request 2 → 🟢 Groq Llama 3.1 8B
Request 3 → 🟣 Gemini Flash Lite
Request 4 → 🟢 Groq Llama 3.1 8B
Request 5 → 🟣 Gemini Flash Lite
Request 6 → 🟢 Groq Llama 3.1 8B
Request 7 → 🟣 Gemini Flash Lite
Request 8 → 🟢 Groq Llama 3.1 8B

🎯 Distribution:
   🟣 Gemini Flash Lite: ████ (4)
   🟢 Groq Llama 3.1 8B: ████ (4)


### 🎯 Strategy 2: latency-based-routing — always pick the fastest one

The router learns which deployment responds fastest and prefers it on future requests.

In [40]:
import os
import time
from litellm import Router

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 OpenAI GPT-4o-mini"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama-3.3"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"
)

print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

Req   Deployment                      Latency   
--------------------------------------------------
#1    🟢 Groq Llama-3.3                   181 ms
#2    🟢 Groq Llama-3.3                   181 ms


InternalServerError: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.. Received Model Group=chat
Available Model Group Fallbacks=None LiteLLM Retried: 2 times, LiteLLM Max Retries: 2

### 🎯 Strategy 3: cost-based-routing — prefer the cheapest deployment

This is useful when quality is acceptable across models and cost matters most.

In [41]:
import os
from litellm import Router

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o",             # ~$2.50/M input tokens
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 GPT-4o (premium)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",        # ~$0.15/M input tokens
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 GPT-4o-mini (cheap)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",   # ~$0.05/M
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama (cheapest)"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

for i in range(5):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

Request 1 → 🟢 Groq Llama (cheapest)


InternalServerError: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.. Received Model Group=chat
Available Model Group Fallbacks=None LiteLLM Retried: 2 times, LiteLLM Max Retries: 2

## 🏆: Production Best Practices

Before you ship a real LLM Gateway, lock these down:

| # | Practice | Why |
|---|----------|-----|
| 1 | **Use Redis caching, not in-memory** | Survives restarts, shared across replicas |
| 2 | **Set per-user rate limits** | Stop one bad actor from burning the budget |
| 3 | **Log to an observability backend** | Langfuse, Helicone, Arize, or your own DB |
| 4 | **Use a master key + virtual keys per team** | Audit trail and chargeback |
| 5 | **Pin model versions** in config | Avoid silent provider-side regressions |
| 6 | **Always set timeouts and `num_retries`** | Don't let hung calls block users |
| 7 | **Configure PII redaction** | Strip emails, phones, SSNs before logging |
| 8 | **Health-check each deployment** | Auto-disable unhealthy providers |
| 9 | **Run the proxy in K8s with HPA** | Scale with traffic |
| 10 | **Version your `config.yaml` in Git** | Treat gateway config as code |

## 🆚 Part: Popular LLM Gateways Compared

| Gateway | Type | Best For |
|---------|------|----------|
| **LiteLLM** | Open-source | The Swiss army knife — 100+ providers, easy to self-host |
| **Portkey** | SaaS / OSS | Strong observability dashboard, prompt management |
| **Helicone** | SaaS / OSS | Drop-in OpenAI proxy with great logging UI |
| **Cloudflare AI Gateway** | SaaS | Already on Cloudflare? One-click setup, edge caching |
| **Kong AI Gateway** | Enterprise | Built on Kong's API gateway, deep enterprise features |
| **OpenRouter** | SaaS | Easy access to 100+ models with one billing account |

For most teams, **LiteLLM is the right starting point** — open source, full control, runs anywhere.